# 🌍 World Bank 2015 — Global Development Analysis
**Dataset:** 2015 World Bank Data by Nation and Region  
**Author:** dasanurag97-maker  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## 📌 Key Questions This Analysis Answers
1. Which regions have the highest and lowest GDP?
2. Is there a relationship between GDP and Internet usage?
3. How does Life Expectancy vary across regions?
4. Which countries have the lowest female literacy rates?
5. What is the relationship between population growth and GDP?
6. Which regions lead in exports?
7. Where does India stand globally across all indicators?

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded successfully!')

## 📂 Step 2: Load & Explore the Data

In [ ]:
# Load all three sheets
df = pd.read_excel('data/2015 World Bank data by nation and region.xlsx',
                   sheet_name='Countries in Alpha Order')
df_region = pd.read_excel('data/2015 World Bank data by nation and region.xlsx',
                          sheet_name='Region Totals')

# Rename columns for easier use
df.columns = ['Country', 'Region', 'Code', 'GDP', 'Population',
              'Pop_Growth', 'Internet_Users', 'Urban_Pop_Pct',
              'Life_Expectancy', 'Female_Literacy', 'Exports_GDP_Pct']

df_region.columns = ['Region', 'Region_Code', 'Code', 'GDP', 'Population',
                     'Pop_Growth', 'Internet_Users', 'Urban_Pop_Pct',
                     'Life_Expectancy', 'Female_Literacy', 'Exports_GDP_Pct']

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 🔍 Step 3: Data Cleaning & Missing Value Check

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': missing_pct})
print(missing_df)

# Drop rows with no region code (aggregate/non-country rows)
df_clean = df.dropna(subset=['Region']).copy()
print(f'\n✅ Clean dataset: {df_clean.shape[0]} countries')

In [ ]:
# Visualize missing data
plt.figure(figsize=(12, 5))
missing_pct_clean = missing_pct[missing_pct > 0]
bars = plt.bar(missing_pct_clean.index, missing_pct_clean.values, color=sns.color_palette('Reds_r', len(missing_pct_clean)))
plt.title('Missing Data by Column (%)', fontsize=15, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Missing (%)')
for bar, val in zip(bars, missing_pct_clean.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('images/missing_data.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q1: Which regions have the highest GDP?

In [ ]:
region_map = {
    'AF': 'Sub-Saharan Africa', 'AU': 'Australasia', 'EA': 'East Asia',
    'EU': 'Europe', 'LA': 'Latin America', 'MA': 'Middle East/N.Africa',
    'ME': 'Middle East', 'NA': 'North America', 'SA': 'South Asia',
    'SE': 'Southeast Asia'
}

region_gdp = df_clean.groupby('Region')['GDP'].sum().sort_values(ascending=False).reset_index()
region_gdp['Region_Name'] = region_gdp['Region'].map(region_map).fillna(region_gdp['Region'])
region_gdp['GDP_Trillion'] = region_gdp['GDP'] / 1e12

plt.figure(figsize=(13, 6))
colors = sns.color_palette('Blues_r', len(region_gdp))
bars = plt.barh(region_gdp['Region_Name'], region_gdp['GDP_Trillion'], color=colors)
plt.xlabel('GDP (Trillion USD PPP)')
plt.title('🌍 Total GDP by World Region (2015)', fontsize=15, fontweight='bold')
for bar, val in zip(bars, region_gdp['GDP_Trillion']):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'${val:.1f}T', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('images/gdp_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q2: Is there a relationship between GDP and Internet Usage?

In [ ]:
df_scatter = df_clean.dropna(subset=['GDP', 'Internet_Users'])
df_scatter['GDP_Billions'] = df_scatter['GDP'] / 1e9

plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    df_scatter['GDP_Billions'],
    df_scatter['Internet_Users'],
    c=df_scatter['Life_Expectancy'], cmap='RdYlGn',
    alpha=0.7, s=80, edgecolors='grey', linewidths=0.4
)
plt.colorbar(scatter, label='Life Expectancy (years)')

# Highlight India
india = df_clean[df_clean['Country'] == 'India']
if not india.empty:
    plt.annotate('🇮🇳 India',
                 xy=(india['GDP'].values[0]/1e9, india['Internet_Users'].values[0]),
                 xytext=(india['GDP'].values[0]/1e9 + 1000, india['Internet_Users'].values[0] + 5),
                 fontsize=11, color='blue', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='blue'))

plt.xscale('log')
plt.xlabel('GDP (Billions USD, log scale)')
plt.ylabel('Internet Users (per 100 people)')
plt.title('💻 GDP vs Internet Usage (colored by Life Expectancy)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/gdp_vs_internet.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q3: How does Life Expectancy vary across regions?

In [ ]:
df_life = df_clean.dropna(subset=['Life_Expectancy', 'Region'])
df_life['Region_Name'] = df_life['Region'].map(region_map).fillna(df_life['Region'])

order = df_life.groupby('Region_Name')['Life_Expectancy'].median().sort_values(ascending=False).index

plt.figure(figsize=(13, 7))
sns.boxplot(data=df_life, x='Region_Name', y='Life_Expectancy',
            order=order, palette='Set2')
plt.xticks(rotation=30, ha='right')
plt.title('❤️ Life Expectancy Distribution by Region (2014)', fontsize=14, fontweight='bold')
plt.xlabel('Region')
plt.ylabel('Life Expectancy (years)')
plt.tight_layout()
plt.savefig('images/life_expectancy_region.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q4: Which countries have the lowest female literacy rates?

In [ ]:
df_lit = df_clean.dropna(subset=['Female_Literacy']).nsmallest(15, 'Female_Literacy')

plt.figure(figsize=(12, 7))
colors = ['#d73027' if v < 40 else '#fc8d59' if v < 60 else '#fee090'
          for v in df_lit['Female_Literacy']]
bars = plt.barh(df_lit['Country'], df_lit['Female_Literacy'], color=colors)
plt.axvline(x=50, color='black', linestyle='--', linewidth=1, label='50% mark')
plt.xlabel('Female Literacy Rate (%)')
plt.title('📚 Bottom 15 Countries — Female Literacy Rate (2015)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, df_lit['Female_Literacy']):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10)
plt.legend()
plt.tight_layout()
plt.savefig('images/female_literacy_bottom15.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q5: Top 10 most populous countries and their Internet access

In [ ]:
top10_pop = df_clean.dropna(subset=['Population']).nlargest(10, 'Population')

fig, ax1 = plt.subplots(figsize=(13, 6))

x = range(len(top10_pop))
bars = ax1.bar(x, top10_pop['Population']/1e6, color=sns.color_palette('Blues', 10), label='Population (M)')
ax1.set_ylabel('Population (Millions)', color='steelblue')
ax1.set_xticks(list(x))
ax1.set_xticklabels(top10_pop['Country'], rotation=30, ha='right')

ax2 = ax1.twinx()
ax2.plot(list(x), top10_pop['Internet_Users'].values, 'ro-', linewidth=2, markersize=8, label='Internet Users %')
ax2.set_ylabel('Internet Users (per 100 people)', color='red')

plt.title('👥 Top 10 Populous Countries vs Internet Access', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.savefig('images/population_vs_internet.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q6: Which regions lead in Exports (% of GDP)?

In [ ]:
df_exp = df_clean.dropna(subset=['Exports_GDP_Pct', 'Region'])
df_exp['Region_Name'] = df_exp['Region'].map(region_map).fillna(df_exp['Region'])
region_exports = df_exp.groupby('Region_Name')['Exports_GDP_Pct'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
colors = sns.color_palette('YlOrRd_r', len(region_exports))
bars = plt.bar(region_exports.index, region_exports.values * 100, color=colors)
plt.xticks(rotation=30, ha='right')
plt.ylabel('Average Exports (% of GDP)')
plt.title('📦 Average Exports as % of GDP by Region (2015)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, region_exports.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val*100:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('images/exports_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

## ❓ Q7: Correlation Heatmap — How are all indicators related?

In [ ]:
cols = ['GDP', 'Population', 'Pop_Growth', 'Internet_Users',
        'Life_Expectancy', 'Female_Literacy', 'Exports_GDP_Pct']
corr = df_clean[cols].corr()

labels = ['GDP', 'Population', 'Pop Growth', 'Internet Users',
          'Life Expectancy', 'Female Literacy', 'Exports %GDP']

plt.figure(figsize=(10, 8))
mask = corr.isnull()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5,
            xticklabels=labels, yticklabels=labels)
plt.title('🔥 Correlation Heatmap — World Bank Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 🇮🇳 Bonus: Where does India stand globally?

In [ ]:
india = df_clean[df_clean['Country'] == 'India'].iloc[0]
indicators = ['GDP', 'Internet_Users', 'Life_Expectancy', 'Female_Literacy', 'Exports_GDP_Pct']
labels_india = ['GDP (B$)', 'Internet Users %', 'Life Expectancy', 'Female Literacy %', 'Exports % GDP']

print('🇮🇳 India — Global Rankings (2015)')
print('='*50)
for col, label in zip(indicators, labels_india):
    val = india[col]
    col_data = df_clean[col].dropna()
    rank = (col_data > val).sum() + 1
    total = len(col_data)
    if col == 'GDP':
        print(f'{label}: ${val/1e9:.0f}B  |  Rank: {rank}/{total}')
    elif col == 'Exports_GDP_Pct':
        print(f'{label}: {val*100:.1f}%  |  Rank: {rank}/{total}')
    else:
        print(f'{label}: {val:.1f}  |  Rank: {rank}/{total}')

## ✅ Conclusion & Key Insights

### 🔑 Key Findings:
1. **GDP:** Europe and East Asia dominate global GDP, while Sub-Saharan Africa lags significantly behind.
2. **Internet Access:** Strongly correlated with GDP — wealthier nations have significantly higher internet penetration.
3. **Life Expectancy:** North America and Europe have the highest life expectancy (75–85 years); Sub-Saharan Africa has the lowest.
4. **Female Literacy:** Several African and South Asian countries have critically low female literacy rates below 40%.
5. **Exports:** Southeast Asia and Australasia regions are the most export-driven economies.
6. **India:** Ranks among the top 10 economies by GDP but lags in internet access and female literacy, indicating significant growth potential.

### 💡 Recommendations:
- **Investment in education** (especially female literacy) could be the biggest lever for developing economies.
- **Digital infrastructure** investment shows strong correlation with economic growth.
- **Export-led growth** strategies have proven successful in Southeast Asia and could be a model for other regions.